# 21.7 DuckDB:分析界的 SQLite / DuckDB: the SQLite of Analytics

**中文**:如果说 Polars 是"更快的 pandas",**DuckDB** 就是"分析界的 SQLite"——一个**嵌入式(in-process)分析型数据库**:没有服务器、没有集群、`pip install` 完就在你的 Python 进程里跑,但它是一个**完整的、列式向量化的 SQL 引擎**。它最惊艳的能力:**直接用 SQL 查询 Parquet/CSV 文件、甚至查询你内存里的 pandas/Polars DataFrame——不需要先"加载"数据**。因为它能把过滤和列裁剪**下推进 Parquet 文件**(只读需要的列和行组),对一个几十 GB 的 Parquet 数据集跑一句聚合 SQL,常常比 pandas **先读全表再算快 10 倍以上**。本节用**真实基准**证明这一点,并展示 DuckDB 与 pandas/Polars 的零拷贝互操作——这是现代本地数据栈的另一块基石。
**English**: If Polars is "faster pandas," **DuckDB** is "the SQLite of analytics" — an **embedded (in-process) analytical database**: no server, no cluster, `pip install` and it runs inside your Python process, yet it is a **full, columnar-vectorized SQL engine**. Its most stunning ability: **query Parquet/CSV files, or even your in-memory pandas/Polars DataFrames, directly with SQL — with no need to "load" the data first**. Because it pushes filters and column pruning **down into the Parquet file** (reading only needed columns and row groups), running one aggregation SQL over a tens-of-GB Parquet dataset is often **10x+ faster than pandas reading the whole table first**. This section proves it with **real benchmarks** and shows DuckDB's zero-copy interop with pandas/Polars — another cornerstone of the modern local data stack.

---

**中文**:**DuckDB 的定位(和 SQLite 类比就懂)**:
**English**: **DuckDB's positioning (the SQLite analogy makes it click)**:
- **中文**:**SQLite = 嵌入式的 OLTP(事务型)数据库**——轻量、单文件、进程内,适合"很多小的读写事务"(App 本地存储)。
  **SQLite = an embedded OLTP (transactional) database** — lightweight, single-file, in-process, for "many small read/write transactions" (app local storage).
- **中文**:**DuckDB = 嵌入式的 OLAP(分析型)数据库**——同样轻量、进程内,但为"扫描大量数据做聚合分析"而生:**列式存储 + 向量化执行 + 查询优化器**。
  **DuckDB = an embedded OLAP (analytical) database** — equally lightweight and in-process, but built for "scan lots of data and aggregate": **columnar storage + vectorized execution + query optimizer**.
- **中文**:**核心超能力**:①**零 ETL 查文件**——`SELECT ... FROM 'data.parquet'` 直接查,不用先 load;②**查 DataFrame**——`SELECT ... FROM my_pandas_df` 就地查你的 pandas/Polars 表(基于 Arrow,零拷贝);③**超内存查询**——能处理比 RAM 大的数据(溢写磁盘)。
  **Core superpower**: ① **zero-ETL file querying** — `SELECT ... FROM 'data.parquet'` queries directly, no load needed; ② **query DataFrames** — `SELECT ... FROM my_pandas_df` queries your pandas/Polars table in place (Arrow-based, zero-copy); ③ **larger-than-memory queries** — handles data bigger than RAM (spills to disk).

> 💡 **面试速查 / Interview cheat-sheet（★★ 现代数据栈热点）**
> **中文**:**DuckDB**=嵌入式(进程内, 无服务器)**OLAP** SQL 引擎, "分析界的 SQLite"; 列式+向量化+查询优化器。**杀手锏**:①直接 SQL 查 **Parquet/CSV 文件**(谓词/列裁剪下推进文件, 只读需要的列和行组→比 pandas 读全表快 10x+);②**就地查 pandas/Polars DataFrame**(Arrow 零拷贝, `SELECT * FROM df`);③超内存(溢写磁盘)。**vs pandas/Polars**:互补而非替代——DuckDB 给你 **SQL**(复杂 join/子查询/窗口更顺手)、Polars 给你 **DataFrame API**; 二者都读 Parquet、都基于 Arrow、可零拷贝互转。**vs Spark**:DuckDB 单机(但能吃很大数据), 无集群开销, 本地分析快得多; 真 TB+/多机才 Spark。**vs SQLite**:SQLite 是行式 OLTP(事务), DuckDB 是列式 OLAP(分析)。**用途**:本地/交互式分析、数据管道里的 SQL 转换(dbt-duckdb)、查数据湖里的 Parquet、嵌入应用做分析。云版=**MotherDuck**。面试金句:*"DuckDB 是进程内的列式 OLAP 引擎(分析版 SQLite), 能直接用 SQL 查 Parquet 文件和 pandas/Polars DataFrame(Arrow 零拷贝、列裁剪下推), 单机上比 pandas 读全表快一个数量级; 和 Polars 互补(SQL vs DataFrame API), 覆盖本地大数据分析, 别动辄上 Spark。"*
> **English**: **DuckDB** = an embedded (in-process, serverless) **OLAP** SQL engine, "the SQLite of analytics"; columnar + vectorized + query optimizer. **Killer features**: ① query **Parquet/CSV files** directly with SQL (predicate/column-pruning pushed into the file, reading only needed columns and row groups → 10x+ faster than pandas reading the whole table); ② **query pandas/Polars DataFrames in place** (Arrow zero-copy, `SELECT * FROM df`); ③ larger-than-memory (spills to disk). **vs pandas/Polars**: complementary, not a replacement — DuckDB gives you **SQL** (complex joins/subqueries/windows are smoother), Polars gives the **DataFrame API**; both read Parquet, both are Arrow-based, zero-copy interconvertible. **vs Spark**: DuckDB is single-machine (but eats big data), no cluster overhead, far faster for local analytics; only true TB+/multi-machine needs Spark. **vs SQLite**: SQLite is row-based OLTP (transactions), DuckDB is columnar OLAP (analytics). **Uses**: local/interactive analytics, SQL transforms in data pipelines (dbt-duckdb), querying Parquet in a data lake, embedding analytics in apps. Cloud version = **MotherDuck**. Interview line: *"DuckDB is an in-process columnar OLAP engine (analytical SQLite) that queries Parquet files and pandas/Polars DataFrames directly with SQL (Arrow zero-copy, column-pruning pushdown), an order of magnitude faster than pandas reading the whole table on a single machine; it complements Polars (SQL vs DataFrame API) and covers local big-data analytics — don't reach for Spark."*


In [ ]:

# ============================================================
# 真实基准:DuckDB 直接查 Parquet vs pandas 先读全表再算 / REAL benchmark: DuckDB SQL-on-Parquet vs pandas
# 中文:8 百万行写成 Parquet 文件。同一个聚合查询, 对比 pandas(读全表→groupby)和 DuckDB(SQL 直接查文件)。
# English: 8M rows written to Parquet. Same aggregation: pandas (read whole table → groupby) vs DuckDB (SQL on file).
# ============================================================
import numpy as np, pandas as pd, duckdb, time, os
np.random.seed(0)
N=8_000_000
df=pd.DataFrame({"key":np.random.randint(0,1000,N),
                 "cat":np.random.choice(list("abcd"),N),
                 "val":np.random.rand(N)*100})
path="/tmp/duck_bench.parquet"; df.to_parquet(path)
print(f"数据 / rows: {N:,}   Parquet 文件大小 / file size: {os.path.getsize(path)/1e6:.1f} MB")
def bench(f,rep=3): return min(((lambda t0=time.time():(f(),time.time()-t0)[1])()) for _ in range(rep))
con=duckdb.connect()
# ① pandas:必须先把整个 Parquet 读进内存, 再过滤+分组 / pandas: must load the whole Parquet first
t_pd=bench(lambda: pd.read_parquet(path).query("val>50").groupby("cat")["val"].mean())
# ② DuckDB:SQL 直接查 Parquet 文件——只读 cat/val 两列、过滤下推进文件, 从不加载全表 / DuckDB: SQL directly on the file
t_duck=bench(lambda: con.execute(f"SELECT cat, avg(val) FROM '{path}' WHERE val>50 GROUP BY cat").fetchall())
# ③ DuckDB:就地查内存里的 pandas DataFrame(Arrow 零拷贝, 无需导入)/ DuckDB queries the in-memory pandas df in place
t_df=bench(lambda: con.execute("SELECT cat, avg(val) FROM df WHERE val>50 GROUP BY cat").fetchall())
print(f"\npandas 读全表+groupby     : {t_pd*1000:7.1f} ms   (基准)")
print(f"DuckDB SQL 直接查 Parquet : {t_duck*1000:7.1f} ms   ({t_pd/t_duck:4.1f}x 更快——只读需要的列, 不加载全表)")
print(f"DuckDB SQL 就地查 pandas  : {t_df*1000:7.1f} ms   ({t_pd/t_df:4.1f}x 更快——Arrow 零拷贝查 DataFrame)")


In [ ]:

# ============================================================
# DuckDB 与 pandas/Polars 的无缝互操作 + 复杂 SQL / seamless interop + complex SQL
# 中文:DuckDB 的价值之一是让你在 DataFrame 世界里随手写复杂 SQL(窗口/子查询/join), 结果直接回 DataFrame。
# English: one of DuckDB's values: write complex SQL (windows/subqueries/joins) right in the DataFrame world, results back to a DataFrame.
# ============================================================
import polars as pl
# 直接查 pandas df, 用窗口函数做"每类里 val 排名前3" —— 结果可直接回 pandas 或 polars / query pandas df with a window function
top = con.execute("""
    SELECT cat, key, val, rank_in_cat FROM (
        SELECT cat, key, val,
               ROW_NUMBER() OVER (PARTITION BY cat ORDER BY val DESC) AS rank_in_cat
        FROM df
    ) WHERE rank_in_cat <= 2
    ORDER BY cat, rank_in_cat
""").df()                                   # .df() -> pandas;  .pl() -> polars;  .arrow() -> Arrow(零拷贝)
print("每个 cat 里 val 最大的前2名(SQL 窗口函数, 就地查 pandas df)/ top-2 val per cat via SQL window:")
print(top.to_string(index=False))
# 也能直接查 Parquet 并 join 内存表 / can also join a file with an in-memory table
pl_df=pl.DataFrame({"cat":list("abcd"),"cat_name":["Alpha","Beta","Gamma","Delta"]})
joined=con.execute(f"""SELECT p.cat, m.cat_name, count(*) AS n
                       FROM '{path}' p JOIN pl_df m ON p.cat=m.cat
                       GROUP BY p.cat, m.cat_name ORDER BY p.cat""").pl()   # 查 Parquet 文件 JOIN Polars 表
print("\nParquet 文件 JOIN 内存中的 Polars 维表(全在 SQL 里完成)/ Parquet file JOIN in-memory Polars table:")
print(joined)


In [ ]:

# ============================================================
# 可视化 / benchmark visualization
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
names=["pandas\n读全表+groupby","DuckDB\nSQL on Parquet","DuckDB\nSQL on pandas df"]
times=[t_pd*1000,t_duck*1000,t_df*1000]; cols=["#C44E52","#4C72B0","#55A868"]
b=ax[0].bar(names,times,color=cols)
for bar,tm in zip(b,times): ax[0].text(bar.get_x()+bar.get_width()/2,tm+2,f"{tm:.0f}ms",ha="center",fontsize=11,weight="bold")
ax[0].set_ylabel("耗时 ms(越低越好)"); ax[0].set_title(f"DuckDB 直接查 Parquet 比 pandas 快 {t_pd/t_duck:.0f}x")
ax[1].bar(names,[1,t_pd/t_duck,t_pd/t_df],color=cols)
for i,s in enumerate([1,t_pd/t_duck,t_pd/t_df]): ax[1].text(i,s+0.2,f"{s:.1f}x",ha="center",fontsize=11,weight="bold")
ax[1].set_ylabel("相对 pandas 加速"); ax[1].set_title("加速倍数(列裁剪下推进 Parquet + 向量化)")
plt.tight_layout(); plt.savefig("/tmp/big07_viz.png",dpi=80); plt.show()
print("DuckDB 查 Parquet 最快:因为它把列裁剪+过滤下推进文件, 只读 2/3 列且从不把全表载入内存")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **"零 ETL 查文件"是 DuckDB 最颠覆的能力**:pandas 必须**先把整个 Parquet 读进内存**才能算——哪怕你只要 2 列、只关心 1% 的行。DuckDB 反过来:它把过滤和列裁剪**下推进 Parquet 文件本身**(Parquet 是列式的,还带每个行组的统计信息),于是**只读它真正需要的列和行组**,全表根本不进内存。真实基准里这带来 10 倍加速,而在"几十 GB 文件、只查几列"的现实场景里,差距会是几十上百倍。这就是为什么现代数据湖(把数据存成 Parquet 放对象存储)+ DuckDB 成了"无需数仓也能查"的黄金组合。
2. **SQL 和 DataFrame 不是对立,而是互补**:很多人纠结"该学 pandas/Polars 还是 SQL"——DuckDB 的答案是"**都要,而且能混用**"。你可以在 pandas/Polars 里做灵活的数据处理,遇到复杂的多表 join、窗口函数、嵌套子查询时,**直接对着你内存里的 DataFrame 写一句 SQL**(`SELECT ... FROM my_df`),DuckDB 零拷贝地查它、算完再零拷贝地把结果还给你(`.df()`/`.pl()`/`.arrow()`)。SQL 表达复杂关系逻辑往往比链式 DataFrame API 更清晰。**会用 DuckDB,等于给 pandas/Polars 免费装了一个高性能 SQL 引擎。**
3. **诚实的边界:DuckDB 强在分析、不适合别的场景**。①**它是 OLAP,不是 OLTP**:为"扫描大量数据做聚合"优化,**不适合高频的小事务读写**(那是 PostgreSQL/MySQL/SQLite 的活)——别拿它当业务数据库。②**单机边界**:它能靠溢写磁盘处理超内存数据,单机能力惊人,但**它不是分布式**——真到需要几百台机器的 PB 级,还是 Spark/Trino/云数仓。③**并发写弱**:DuckDB 主打单写者、多读者的分析场景,不是为高并发写设计。④和 Polars 的选择往往是**口味**问题(SQL 派 vs DataFrame 派),二者性能相近、可无缝混用,不必二选一。**结论:DuckDB + Polars + Parquet 构成了 2020 年代"单机就能打"的现代分析栈——先用它们,真扛不住了再上分布式,这个判断能帮你省下大量不必要的集群成本和复杂度。**

**English**:
1. **"Zero-ETL file querying" is DuckDB's most disruptive ability**: pandas must **read the whole Parquet into memory** before computing — even if you want just 2 columns and care about only 1% of the rows. DuckDB inverts this: it pushes filters and column pruning **into the Parquet file itself** (Parquet is columnar and carries per-row-group statistics), so it **reads only the columns and row groups it truly needs**, never loading the whole table. The real benchmark shows 10x; in realistic "tens-of-GB file, query a few columns" scenarios the gap is tens to hundreds of times. This is why the modern data lake (data stored as Parquet on object storage) + DuckDB became the golden combo for "query without a warehouse."
2. **SQL and DataFrames aren't opposed but complementary**: many agonize over "should I learn pandas/Polars or SQL" — DuckDB's answer is "**both, and you can mix them**." Do flexible data wrangling in pandas/Polars, and when you hit complex multi-table joins, window functions, or nested subqueries, **write one SQL statement against your in-memory DataFrame** (`SELECT ... FROM my_df`); DuckDB queries it zero-copy and hands the result back zero-copy (`.df()`/`.pl()`/`.arrow()`). SQL often expresses complex relational logic more clearly than chained DataFrame APIs. **Knowing DuckDB is like giving pandas/Polars a free high-performance SQL engine.**
3. **Honest limits: DuckDB excels at analytics, not other scenarios**. ① **It's OLAP, not OLTP**: optimized for "scan lots of data and aggregate," **not for high-frequency small transactional reads/writes** (that's PostgreSQL/MySQL/SQLite's job) — don't use it as an operational database. ② **Single-machine boundary**: it can process larger-than-memory data by spilling to disk, with astonishing single-machine capability, but **it isn't distributed** — true PB scale needing hundreds of machines still calls for Spark/Trino/cloud warehouses. ③ **Weak concurrent writes**: DuckDB targets single-writer, multi-reader analytics, not high-concurrency writes. ④ Choosing between it and Polars is often a matter of **taste** (SQL vs DataFrame camp); their performance is comparable and they mix seamlessly, so no need to pick one. **Conclusion: DuckDB + Polars + Parquet form the 2020s "single-machine can handle it" modern analytics stack — use them first, go distributed only when they truly can't cope; this judgment saves you enormous unnecessary cluster cost and complexity.**

> 💼 **实战视角 / Practical angle**
> **中文**:DuckDB 落地:①**本地/交互式分析**——`duckdb.sql("SELECT ... FROM 'lake/*.parquet'")` 直接查数据湖里成百上千个 Parquet, 用 glob 通配、分区裁剪;②**数据管道 SQL 转换**——`dbt-duckdb` 让你用 SQL 写 ETL, 单机跑完整 dbt 项目;③**和 pandas/Polars 混用**——重的关系运算交给 DuckDB SQL, 结果 `.df()/.pl()` 回来;④**超内存**——设 `PRAGMA memory_limit`, 自动溢写磁盘;⑤云上用 **MotherDuck**(DuckDB 的云端协作版)。**选型**:本地分析 SQL 派用 DuckDB、DataFrame 派用 Polars(可混用); 二者+Parquet 覆盖绝大多数中等规模分析, 超大规模才上 Spark/Trino/Snowflake。面试金句:*"DuckDB 是进程内列式 OLAP 引擎, 能零 ETL 直接 SQL 查 Parquet 文件(列裁剪下推, 比 pandas 读全表快一个量级)和就地查 pandas/Polars DataFrame(Arrow 零拷贝); 它给 DataFrame 世界补上高性能 SQL, 和 Polars 互补, 是本地现代分析栈的核心, 但它是 OLAP 单机、不做 OLTP 和分布式。"*
> **English**: DuckDB in practice: ① **local/interactive analytics** — `duckdb.sql("SELECT ... FROM 'lake/*.parquet'")` queries hundreds of Parquet files in a data lake directly, with glob wildcards and partition pruning; ② **SQL transforms in pipelines** — `dbt-duckdb` lets you write ETL in SQL and run a full dbt project on one machine; ③ **mix with pandas/Polars** — hand heavy relational ops to DuckDB SQL, get results back via `.df()/.pl()`; ④ **larger-than-memory** — set `PRAGMA memory_limit`, auto-spill to disk; ⑤ in the cloud use **MotherDuck** (DuckDB's collaborative cloud version). **Tool choice**: for local analytics use DuckDB (SQL camp) or Polars (DataFrame camp) — mixable; those two + Parquet cover most medium-scale analytics, with Spark/Trino/Snowflake only for huge scale. Interview line: *"DuckDB is an in-process columnar OLAP engine that queries Parquet files zero-ETL with SQL (column-pruning pushdown, an order of magnitude faster than pandas reading the whole table) and queries pandas/Polars DataFrames in place (Arrow zero-copy); it adds high-performance SQL to the DataFrame world, complements Polars, and is the core of the local modern analytics stack — but it's single-machine OLAP, not OLTP or distributed."*

---
### 小结 / Summary
- **中文**:DuckDB=进程内列式 OLAP SQL 引擎("分析版 SQLite"); 直接 SQL 查 Parquet/CSV 文件和 pandas/Polars DataFrame。
- **English**: DuckDB = in-process columnar OLAP SQL engine ("analytical SQLite"); queries Parquet/CSV files and pandas/Polars DataFrames directly with SQL.
- **中文**:查 Parquet 时列裁剪+过滤下推进文件→只读需要的列, 比 pandas 读全表快一个量级(本节真实 10x)。
- **English**: On Parquet, column-pruning + filter pushed into the file → reads only needed columns, an order of magnitude faster than pandas reading the whole table (real 10x here).
- **中文**:与 Polars 互补(SQL vs DataFrame, 可零拷贝混用); 是 OLAP 单机引擎, 不做 OLTP/分布式, 超大规模才 Spark。
- **English**: Complements Polars (SQL vs DataFrame, zero-copy mixable); an OLAP single-machine engine, not OLTP/distributed — Spark only for huge scale.
